# Week 2, day 4 (afternoon) — More practice 06 SOLUTIONS: strings, conditionals and loops

Every cell below was executed on the same Python the lab ships; the quoted
output is real.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# More practice 06 — Strings, conditionals and loops. Run this once.
log_line = "2026-08-26,ERROR,payment-service,timeout after 30s"
csv_rows = [
    "ada,36,Toronto",
    "bo,24,Ottawa",
    "cai,41,Montreal",
]
title = "  the Rest is Naming Things  "
basket = [12.99, 4.50, 30.00, 7.25, 99.99]

print("log_line:", log_line)
print("csv_rows:", csv_rows)
print("title:   ", repr(title))
print("basket:  ", basket)

PART A — Taking strings apart

### Question 1

Warm-up — one expression. -> `62.0`, twice.

`3 * 12.50 + 2 * 7.25 + 10.00`. Multiplication binds tighter than addition,
so the brackets you might have reached for are not needed.

The result is `62.0` rather than `62` because the inputs were floats. Note
`round(..., 2)` did not add trailing zeros — rounding produces a NUMBER,
and numbers have no opinion about decimal places. Formatting for display is
a separate job.

In [ ]:
print(3 * 12.50 + 2 * 7.25 + 10.00)
print(round(3 * 12.50 + 2 * 7.25 + 10.00, 2))

### Question 2

String methods, none of them destructive. -> the trimmed string, then lower case, then upper case, then title case, then the original still `'  the Rest is Naming Things  '`.

Every one of these returns a NEW string, which is why `title` still has its
two leading and two trailing spaces at the end. Strings are immutable,
exactly like tuples — `title[0] = "T"` raises the same `TypeError` you saw
in worksheet 02 Q5.

The practical consequence: `title.strip()` on a line by itself does
nothing. You have to use the value it returns or assign it back.

In [ ]:
print(title.strip())
print(title.strip().lower())
print(title.strip().upper())
print(title.strip().title())

print(repr(title))   # unchanged -- strings are immutable

### Question 3

Splitting a log line. -> `['2026-08-26', 'ERROR', 'payment-service', 'timeout after 30s']`, then `4`, then the date, severity and service.

Unpacking four fields into four names only works because `split` produced
exactly four. One stray comma inside the message and this raises
`ValueError: too many values to unpack` — and free-text fields are exactly
where stray commas live.

That is the reason real CSV files need a proper parser rather than
`.split(",")`. This works here because the data is well behaved, which is
not a guarantee you get in production.

In [ ]:
fields = log_line.split(",")
print(fields)
print(len(fields))

date, severity, service, message = fields
print(date)
print(severity)
print(service)

### Question 4

Parsing rows into records. -> three sentences, then `33.7`.

`split(",")` always returns STRINGS, so `age` is `"36"` until `int()` is
applied. Skip that conversion and `sum(ages)` would fail outright — or
worse, `+` between strings would have concatenated, exactly as in more-practice
05 Q2.

Building `ages` inside the loop and using it after is the accumulator
pattern again, the same shape as the counting loop in worksheet 08 Q7.

In [ ]:
ages = []
for row in csv_rows:
    name, age, city = row.split(",")
    age = int(age)                 # split() always gives strings
    ages.append(age)
    print(f"{name} is {age} and lives in {city}")

print(round(sum(ages) / len(ages), 1))

PART B — Conditions that combine

### Question 5

Bands with `and`. -> `12.99 mid`, `4.5 bargain`, `30.0 mid`, `7.25 bargain`, `99.99 premium`.

The `price >= 10` half of the middle test is redundant. The `elif` is only
reached when `price < 10` was false, so the price is already known to be at
least 10.

Redundant is not wrong. Some people prefer the explicit version precisely
because it survives someone reordering the branches later, and the cost is
nothing. What matters is knowing which guarantees you are relying on rather
than writing it out by superstition.

In [ ]:
for price in basket:
    if price < 10:
        label = "bargain"
    elif price >= 10 and price < 50:
        label = "mid"
    else:
        label = "premium"
    print(price, label)

# The `price >= 10` half is redundant: the elif is only reached when the
# first test FAILED, so price is already known to be >= 10. Writing it out
# does no harm and some people find it clearer -- but relying on the order
# is what elif is for.

### Question 6

Boolean logic. -> `True`, `False`, `True`.

`or` needs one side to hold; `and` needs both. The second line is `False`
because the severity is `ERROR`, not `INFO` — the `timeout` half never got
a say, since `and` stops as soon as one side fails.

The third line shows the tidier form: `severity in ("ERROR", "CRITICAL")`
says the same thing as a chain of `or`s, and unlike the chain it does not
get longer every time someone adds a severity.

In [ ]:
print(severity == "ERROR" or severity == "CRITICAL")

print(severity == "INFO" and "timeout" not in message)

# A shorter way to write the first: severity in ("ERROR", "CRITICAL")
print(severity in ("ERROR", "CRITICAL"))

PART C — Controlling a loop

### Question 7

break and continue. -> the first four prices then `too expensive, stopping`; then, after the divider, `12.99`, `30.0`, `99.99`.

`break` ended the first loop entirely at `99.99`, so nothing beyond it was
ever examined. `continue` skipped only the items under 10 and carried on —
which is why the second loop saw all five prices and printed three.

A common and expensive bug is reaching for `break` when you meant
`continue`: the loop stops early, the output looks entirely plausible, and
the items you never looked at leave no trace anywhere.

In [ ]:
for price in basket:
    if price > 50:
        print("too expensive, stopping")
        break            # leaves the loop entirely
    print(price)

print("---")

for price in basket:
    if price < 10:
        continue         # skip THIS item, keep looping
    print(price)

### Question 8

A budget loop with two exit conditions. -> buys `12.99`, `4.5` and `30.0`, leaving `2.51`, and reports `items bought: 3`.

The condition has two halves for two unrelated reasons.
`position < len(basket)` stops you indexing past the end of the list;
`budget >= basket[position]` stops you overspending. Drop the first and the
loop raises `IndexError` the moment the money outlasts the basket.

Note it stopped at `7.25` with `2.51` still in hand, even though something
cheaper might sit further along. This walks the basket IN ORDER rather than
shopping optimally — a rule of the exercise, not a bug. Being able to say
confidently which of those it is matters more than the code does.

In [ ]:
budget = 50.00
position = 0
bought = 0

while position < len(basket) and budget >= basket[position]:
    budget = budget - basket[position]
    bought = bought + 1
    print("bought", basket[position], "| left:", round(budget, 2))
    position = position + 1

print("items bought:", bought)

### Question 9

Counting vowels two ways. -> `13` and `13`.

The loop and the comprehension do the same work. The comprehension builds
the list of matching characters and then measures it, where the loop never
stores them — at this size the difference is nothing.

AND `.lower()` IS DOING REAL WORK HERE. `log_line` contains `ERROR`, whose
two vowels are uppercase, so dropping the `.lower()` gives `11` instead of
`13`. A test string in all lower case would have hidden that completely —
which is a good argument for choosing test data that includes the awkward
case.

In [ ]:
vowels = "aeiou"
count = 0
for character in log_line.lower():
    if character in vowels:
        count = count + 1
print(count)

print(len([c for c in log_line.lower() if c in vowels]))

### Question 10

Stretch — FizzBuzz. -> `1`, `2`, `Fizz`, `4`, `Buzz`, `Fizz`, `7`, `8`, `Fizz`, `Buzz`, `11`, `Fizz`, `13`, `14`, `FizzBuzz`, `16`, `17`, `Fizz`, `19`, `Buzz`.

15 IS THE ONLY LINE THAT TESTS YOUR ORDERING, and it is the entire reason
this question gets asked. Put the `n % 3 == 0 and n % 5 == 0` branch last
and 15 prints `Fizz`, because the `elif` chain stops at the first branch
that matches and 15 does divide by 3.

Nineteen of the twenty lines are identical either way. A test that only
checked the first ten numbers would pass a broken implementation without
complaint — which is a lesson about tests at least as much as about `elif`.

In [ ]:
for n in range(1, 21):
    if n % 3 == 0 and n % 5 == 0:    # MUST come first
        print("FizzBuzz")
    elif n % 3 == 0:
        print("Fizz")
    elif n % 5 == 0:
        print("Buzz")
    else:
        print(n)